# Train và đánh giá Transformer tóm tắt văn bản

Notebook này dùng cho Kaggle. Quy tắc quan trọng: **không train trên test**. Test chỉ được dùng sau khi đã train xong để sinh kết quả và tính ROUGE báo cáo.

Mô hình chính: Transformer encoder-decoder tự cài, khoảng **11.47M tham số** với cấu hình `d_model=256`, `layers=4`, `heads=8`, `d_ff=1024`, vocab 16k.

## 1. Copy project từ Kaggle Input sang Working

`/kaggle/input` là read-only, nên cần copy code sang `/kaggle/working` để tạo data/cache/checkpoint/output.

In [ ]:
from pathlib import Path
import shutil

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')

project_src = next(INPUT_ROOT.rglob('summarization_project'))
project_dst = WORK_ROOT / 'summarization_project'

if project_dst.exists():
    shutil.rmtree(project_dst)
shutil.copytree(project_src, project_dst)

req_src = next(INPUT_ROOT.rglob('requirements.txt'), None)
if req_src is not None:
    shutil.copy2(req_src, WORK_ROOT / 'requirements.txt')

print('Copied project from:', project_src)
print('Project working dir:', project_dst)

In [ ]:
%cd /kaggle/working/summarization_project

## 2. Cài thư viện và kiểm tra GPU

In [ ]:
!pip install -q pandas pyarrow sentencepiece rouge-score tqdm pyyaml

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 3. Tìm file train/valid/test

Dùng `find/rglob` để tránh sai đường dẫn do Kaggle dataset slug thay đổi.

In [ ]:
from pathlib import Path

input_root = Path('/kaggle/input')
train_file = next(input_root.rglob('train-00000-of-00001.parquet'))
valid_file = next(input_root.rglob('valid-00000-of-00001.parquet'))
test_file = next(input_root.rglob('test-00000-of-00001*.parquet'))

print('Train:', train_file)
print('Valid:', valid_file)
print('Test :', test_file)

## 4. Chuẩn bị dữ liệu

- Train/valid dùng cho huấn luyện và chọn checkpoint.
- Test chỉ chuẩn hóa và lưu JSONL, **không dùng để train**.

In [ ]:
!python scripts/prepare_data.py \
  --train "{train_file}" \
  --valid "{valid_file}" \
  --source-col article \
  --target-col summary

In [ ]:
!python scripts/prepare_test_data.py \
  --test "{test_file}" \
  --source-col article \
  --target-col summary

## 5. Train tokenizer từ train set

Tokenizer chỉ train trên `train.jsonl`, không train trên valid/test.

In [ ]:
!python scripts/train_tokenizer.py \
  --input data/processed/train.jsonl \
  --vocab-size 16000

## 6. Tokenize và cache train/valid/test

Dùng tokenizer đã train từ train set để encode cả ba split.

In [ ]:
!python scripts/tokenize_cache.py \
  --tokenizer tokenizer/tokenizer_models/summary_bpe.model \
  --splits train valid test \
  --max-source-len 512 \
  --max-target-len 128

In [ ]:
!cat data/cached/tokenization_stats.json

## 7. Kiểm tra số tham số mô hình

Cấu hình báo cáo: khoảng 11.47M tham số.

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/summarization_project/src')
from summarization.dataset import SummaryDataset
from summarization.models import TransformerConfig, SummarizationTransformer

ds = SummaryDataset('data/cached/train_tokenized.pkl')
cfg = TransformerConfig(vocab_size=ds.vocab_size, pad_id=ds.pad_id, d_model=256, num_encoder_layers=4, num_decoder_layers=4, num_heads=8, d_ff=1024)
model = SummarizationTransformer(cfg)
print(f'Parameters: {model.count_parameters():,}')

## 8. Train model trên train/valid

Cell này train 10 epoch. Test không xuất hiện trong train script.

In [ ]:
!python scripts/train_baseline.py \
  --epochs 10 \
  --batch-size 4 \
  --d-model 256 \
  --layers 4 \
  --heads 8 \
  --d-ff 1024 \
  --device cuda

## 9. Đánh giá validation bằng beam search

Validation dùng để báo cáo quá trình chọn mô hình và so sánh decoding.

In [ ]:
!python scripts/generate_summaries.py \
  --checkpoint checkpoints/baseline/best.pt \
  --cache data/cached/valid_tokenized.pkl \
  --processed-jsonl data/processed/valid.jsonl \
  --tokenizer tokenizer/tokenizer_models/summary_bpe.model \
  --method beam \
  --beam-size 4 \
  --length-penalty 0.8 \
  --no-repeat-ngram-size 3 \
  --device cuda \
  --output outputs/valid_predictions_beam.jsonl

In [ ]:
!python scripts/evaluate_rouge.py \
  --predictions outputs/valid_predictions_beam.jsonl \
  --output outputs/valid_rouge_beam.json

!cat outputs/valid_rouge_beam.json

## 10. Ablation nhanh: greedy vs beam

Không train lại. Dùng cùng checkpoint để so sánh tác dụng của beam + length penalty + no-repeat n-gram.

In [ ]:
!python scripts/generate_summaries.py \
  --checkpoint checkpoints/baseline/best.pt \
  --cache data/cached/valid_tokenized.pkl \
  --processed-jsonl data/processed/valid.jsonl \
  --tokenizer tokenizer/tokenizer_models/summary_bpe.model \
  --method greedy \
  --device cuda \
  --output outputs/valid_predictions_greedy.jsonl

!python scripts/evaluate_rouge.py \
  --predictions outputs/valid_predictions_greedy.jsonl \
  --output outputs/valid_rouge_greedy.json

!cat outputs/valid_rouge_greedy.json

## 11. Chạy trên test set

Đây là bước cuối cùng. Model đã train xong và checkpoint đã chọn bằng validation. Test chỉ dùng để sinh dự đoán và tính ROUGE.

In [ ]:
!python scripts/generate_summaries.py \
  --checkpoint checkpoints/baseline/best.pt \
  --cache data/cached/test_tokenized.pkl \
  --processed-jsonl data/processed/test.jsonl \
  --tokenizer tokenizer/tokenizer_models/summary_bpe.model \
  --method beam \
  --beam-size 4 \
  --length-penalty 0.8 \
  --no-repeat-ngram-size 3 \
  --device cuda \
  --output outputs/test_predictions_beam.jsonl

In [ ]:
!python scripts/evaluate_rouge.py \
  --predictions outputs/test_predictions_beam.jsonl \
  --output outputs/test_rouge_beam.json

!cat outputs/test_rouge_beam.json

## 12. Xem vài ví dụ dự đoán test

In [ ]:
import json
from pathlib import Path

pred_path = Path('outputs/test_predictions_beam.jsonl')
for i, line in enumerate(pred_path.open(encoding='utf-8')):
    item = json.loads(line)
    print('=' * 80)
    print('ID:', item['id'])
    print('REFERENCE:', item['reference'][:500])
    print('PREDICTION:', item['prediction'][:500])
    if i == 4:
        break

## 13. Zip kết quả để tải về

In [ ]:
%cd /kaggle/working
!zip -r results_report.zip summarization_project/checkpoints summarization_project/outputs summarization_project/data/processed/test_stats.json summarization_project/data/cached/tokenization_stats.json
!ls -lh results_report.zip